# **Motivation, and Research Questions:**

## **Motivation and Interest**
I chose this dataset because I wrote an essay a few years ago that dove deeply into how social media affects mental health. It is highly engaging to test these concepts directly through hands-on data analysis rather than solely reading published studies. Furthermore, this topic has a strong real-world context in public health and psychology, as understanding digital habits is crucial for addressing the modern youth mental health crisis.

## **Research Questions**
1. *Can we accurately predict the binary "Depression Label" using the provided behavioral and psychological data?*
2. *If so, which of these features holds the most predictive weight?*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

# Load the dataset
df = pd.read_csv('/kaggle/input/datasets/algozee/teenager-menthal-healy/Teen_Mental_Health_Dataset.csv')

# Set a clean visual style for the plots
sns.set_theme(style="whitegrid")

# Set random seed
random_state = 42

print("Setup Complete")

# **1. Exploratory Data Analysis (EDA)**
In this section, we will explore the distributions of our key features and our target outcome (`depression_label`). The goal is to understand the shape of our data and uncover any initial relationships before training our machine learning models.

In [ ]:
df.head(6)

In [ ]:
print('The DataSet Size : ',df.shape)

In [ ]:
# Plots of key numeric features
import warnings

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "figure.dpi": 130, "font.family": "DejaVu Sans"
})


num_cols = ["age", "daily_social_media_hours", "sleep_hours", "screen_time_before_sleep", 
            "academic_performance", "physical_activity", "stress_level", "anxiety_level", "addiction_level"]
colors = ["#3BBFB2", "#9B72AA", "#F0A500", "#E05C5C", "#5B8DB8", "#7FC97F"]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle("Distribution of Key Numeric Features", fontsize=15, fontweight="bold", y=1.02)

for i, ax in enumerate(axes.flatten()):
    col = num_cols[i]
    data = df[col].dropna()
    ax.hist(data, bins=20, color=colors[i % len(colors)], edgecolor="white", alpha=0.88)
    
    # Add Mean and Median lines
    ax.axvline(data.mean(), color="#333333", ls="--", lw=1.2, label=f"Mean: {data.mean():.1f}")
    ax.axvline(data.median(), color="#888888", ls=":", lw=1.2, label=f"Median: {data.median():.1f}")
    
    # Clean up titles and labels
    ax.set(title=col.replace("_", " ").title(), xlabel="Value", ylabel="Count")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate counts and percentages
platform_counts = df['platform_usage'].value_counts()
platform_pcts = df['platform_usage'].value_counts(normalize=True) * 100

plt.figure(figsize=(9, 6))
sns.set_theme(style="whitegrid")

# Create a bar plot
ax = sns.barplot(x=platform_counts.index, y=platform_counts.values, palette='Set2')

# Add text labels with total and percentage
for i, p in enumerate(ax.patches):
    height = p.get_height()
    count = int(platform_counts.values[i])
    pct = platform_pcts.values[i]
    # Position the text slightly above each bar
    ax.text(p.get_x() + p.get_width() / 2., height + 3, 
            f'Total: {count}\n({pct:.1f}%)', 
            ha="center", va="bottom", fontsize=12, fontweight='bold')

# Customize titles and labels
plt.title('Total and Percentage of Social Media Platforms Used', fontweight='bold', pad=15)
plt.xlabel('Social Media Platform')
plt.ylabel('Number of Teenagers')

# Add some headroom so the text labels don't get cut off
plt.ylim(0, max(platform_counts.values) * 1.15) 

# Remove top and right borders for a cleaner look
sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
# Calculate counts and percentages, sorting to keep 0 and 1 in order
dep_counts = df['depression_label'].value_counts().sort_index()
dep_pcts = df['depression_label'].value_counts(normalize=True).sort_index() * 100

plt.figure(figsize=(8, 6))
sns.set_theme(style="whitegrid")

# Create labels mapping
labels = ['No Depression (0)', 'Depression (1)']

# Create a bar plot using custom colors
ax = sns.barplot(x=labels, y=dep_counts.values, palette=['#66b3ff', '#ff9999'])

# Add text labels with total and percentage
for i, p in enumerate(ax.patches):
    height = p.get_height()
    count = int(dep_counts.values[i])
    pct = dep_pcts.values[i]
    # Position the text slightly above each bar
    ax.text(p.get_x() + p.get_width() / 2., height + (max(dep_counts.values) * 0.02), 
            f'Total: {count}\n({pct:.1f}%)', 
            ha="center", va="bottom", fontsize=12, fontweight='bold')

# Customize titles and labels
plt.title('Total and Percentage of Depression Label', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Depression Classification', fontsize=12)
plt.ylabel('Number of Teenagers', fontsize=12)

# Add some headroom so the text labels don't get cut off
plt.ylim(0, max(dep_counts.values) * 1.15) 

# Remove top and right borders for a cleaner look
sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
#Correlation Heatmap of Numeric Features
plt.figure(figsize=(12, 8))

# Filter to only include numeric columns for the correlation math
numeric_cols = df.select_dtypes(include=['int64', 'float64'])
correlation_matrix = numeric_cols.corr()

# Create the heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", 
            linewidths=0.5, vmin=-1, vmax=1)

plt.title('Correlation Heatmap of Behavioral and Mental Health Metrics', fontsize=16)
plt.xticks(rotation=45, ha='right')

plt.show()

# **2. Model Fitting and Training**
In this section, we will train Machine Learning models to predict the `depression_label`. 
First, we must preprocess our categorical text data (like `gender`, `platform_usage`, and `social_interaction_level`) into numerical values using One-Hot Encoding. Then, we will split the data into an 80% training set and a 20% testing set to evaluate our models fairly.

## **Data Prep & Train/Test Split**

In [ ]:
# 1. Prepare the Features (X) and Target (y)
# Drop the target variable to create our feature set
X_raw = df.drop(columns=['depression_label'])
y = df['depression_label']

# 2. Convert categorical text columns into numbers (One-Hot Encoding)
X = pd.get_dummies(X_raw, columns=['gender', 'platform_usage', 'social_interaction_level'], drop_first=True)

# 3. Split the data (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

## **Model 1: Logistic Regression (Baseline)**
We will start with a Logistic Regression model to establish a baseline. This model works well for linear relationships.

In [ ]:
# Initialize the model
log_reg = LogisticRegression(max_iter=1000, random_state=random_state)

# Train the model on the training data
log_reg.fit(X_train, y_train)

# Generate predictions on the unseen test data
y_pred_log = log_reg.predict(X_test)

print("Logistic Regression Model Trained")

## **Model 2 & 3: Decision Tree and Random Forest**
Next, we will use tree-based algorithms to capture complex, non-linear relationships. The Random Forest (an ensemble of many decision trees) will help prevent overfitting and allow us to extract the "Feature Importance" weights to answer our final hypothesis.

In [ ]:
# Initialize the two models
tree_clf = DecisionTreeClassifier(max_depth=5, random_state=random_state)
forest_clf = RandomForestClassifier(n_estimators=100, random_state=random_state)

# Train the two models on the training data
tree_clf.fit(X_train, y_train)
forest_clf.fit(X_train, y_train)

# Generate predictions on the unseen test data
y_pred_tree = tree_clf.predict(X_test)
y_pred_forest = forest_clf.predict(X_test)
print("Tree Based Models Trained")

## **Model Accuracy on Test Data**

In [ ]:
print("--- Model Accuracy on Test Data ---")
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_log):.2%}")
print(f"Decision Tree Accuracy:       {accuracy_score(y_test, y_pred_tree):.2%}")
print(f"Random Forest Accuracy:       {accuracy_score(y_test, y_pred_forest):.2%}")

# **3. Conclusions and Implications**
To explicitly answer our research questions: 
> *"Can we accurately predict the 'Depression Label' using the provided data?"*
> 
> *"If so, which of these features holds the most predictive weight?"*

We can determine this by extracting the feature importance weights directly from our trained Random Forest model. This will allow us to see exactly which variables the model relied on most heavily to predict the `depression_label`.

In [ ]:
# Extract feature importance from the Random Forest model
importances = forest_clf.feature_importances_
feature_names = X.columns

# Create a DataFrame and sort the values
feature_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_df = feature_df.sort_values(by='Importance', ascending=False)

# Plot the Feature Importances
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_df, x='Importance', y='Feature', palette='mako')

plt.title('Random Forest: Feature Importance for Predicting Depression')
plt.xlabel('Relative Predictive Weight')
plt.ylabel('Feature')

plt.show()


## **Final Project Conclusions**
Based on the Exploratory Data Analysis and the Machine Learning models trained above, we can draw the following conclusions to answer the original project questions.

### **Predictive Accuracy and Dataset Implications:**
Yes, we can predict the depression label with high accuracy using the provided data with all three machine learning models on the unseen data. 
* Logistic Regression: 98.75%
* Decision Tree: 99.17%
* Random Forest: 97.92%

However, because real-world psychological metrics rarely yield near-perfect predictive accuracy due to the complex nature of mental health, this exceptionally high performance strongly suggests that the dataset is synthetic. The depression label was likely generated using a direct mathematical formula based on the other features.

### **Feature Importance and Predictive Weight:**
By extracting the feature weights from the Random Forest Classifier, we identified that **Stress Level**, **Daily Social Media Hours**, **Anxiety Level**, and **Sleep Hours** hold the absolute highest predictive weight. 

Interestingly, variables like "Screen Time Before Sleep" carried significantly less direct predictive weight than the resulting psychological symptoms.